In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/creditcard_0.csv")
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12335,21590.0,1.186033,-0.294144,1.019719,-0.565865,-0.928925,-0.295884,-0.705442,0.002490,3.049365,...,-0.124689,0.164870,-0.008048,0.047003,0.411859,-0.697793,0.083593,0.028098,11.85,0
12336,21594.0,-0.714826,0.375221,2.665761,-1.747075,-0.204037,-0.206592,0.330850,-0.099534,2.574038,...,-0.129499,0.200336,-0.335035,0.024292,0.389485,-0.812327,-0.068382,-0.155266,11.85,0
12337,21596.0,-4.729010,0.368596,-3.762416,2.774622,0.057178,4.122468,-0.506988,2.676987,-0.070382,...,-0.191779,-0.444019,-0.853614,1.079924,-0.386361,0.150602,0.597572,-0.328566,277.88,0
12338,21597.0,1.267562,-0.396252,0.563092,-0.894755,-0.578609,0.196242,-0.818880,0.137891,3.017380,...,-0.188144,-0.128655,-0.183524,-0.903625,0.577828,-0.668774,0.055647,0.007328,11.85,0


In [2]:
X = df.drop(["Time", "Class"], axis=1)
y = df["Class"]

X.shape, y.shape

((12340, 29), (12340,))

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=44)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((9255, 29), (3085, 29), (9255,), (3085,))

In [4]:
len(y_train[y_train == 1]), len(y_test[y_test == 1])

(44, 11)

In [5]:
from imblearn.combine import SMOTEENN

smote_enn = SMOTEENN(random_state=0)
X_resampled, y_resampled = smote_enn.fit_resample(X_train, y_train)

X_resampled.shape, y_resampled.shape

((18216, 29), (18216,))

In [6]:
len(y_resampled[y_resampled == 1])

9128

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

model_params = {
    "RandomForestClassifier": {
        "model": RandomForestClassifier(random_state=44), 
        "params": {
            "n_estimators": [10, 100], 
            "criterion": ["gini", "entropy", "log_loss"], 
            "max_depth": [1, 3, 6], 
            "min_samples_split": [1, 2, 4], 
            "min_samples_leaf": [1, 2], 
            "max_leaf_nodes": [16, 32]
        }
    }, 
    "GradientBoostingClassifier": {
        "model": GradientBoostingClassifier(random_state=44), 
        "params": {
            "learning_rate": [0.1, 0.2], 
            "n_estimators": [10, 100], 
            "criterion": ["friedman_mse", "squared_error"], 
            "min_samples_split": [1, 2, 4], 
            "min_samples_leaf": [1, 2], 
            "max_depth": [3, 6], 
            "max_leaf_nodes": [16, 32]
        }
    }, 
    "XGBClassifier": {
        "model": XGBClassifier(random_state=44), 
        "params": {
            "n_estimators": [10, 100], 
            "max_depth": [1, 3, 6], 
            "max_leaves": [16, 32], 
            "max_bin": [16, 32, 64], 
            "grow_policy": ["depthwise", "lostguide"], 
            "learning_rate": [0.1, 0.2], 
            "gamma": [0.1, 0.2], 
            "sampling_method": ["uniform", "gradient_based"], 
            "reg_alpha": [0.1, 0.2], 
            "reg_lambda": [0.1, 0.2]
        }
    }
}

In [ ]:
import mlflow
from sklearn.metrics import classification_report

mlflow.set_tracking_uri("http://127.0.0.1:8080/")
mlflow.set_experiment("fraud-model-training")
mlflow.sklearn.autolog(log_model_signatures=True)

best_f1 = 0
best_recall = 0
best_model_artifact_uri = ""
for model, params in model_params.items():
    with mlflow.start_run(run_name=model):
        gscv = GridSearchCV(estimator=params["model"], param_grid=params["params"], scoring="roc_auc", cv=3)
        try:
            gscv.fit(X_resampled, y_resampled)
            y_pred = gscv.best_estimator_.predict(X_test)
            metrics = classification_report(y_test, y_pred, output_dict=True)["1"]
            mlflow.log_metrics(metrics)

            f1 = metrics["f1-score"]
            recall = metrics["recall"]
            if (f1 >= best_f1) and (recall >= best_recall):
                best_overall_score = f1
                best_recall = recall
                run_info = mlflow.active_run().info
                best_model_artifact_uri = f"runs://{run_info.experiment_id}/{run_info.run_id}/best_estimator"
            
        except:
            print("Training error: skipping logging...")

2026/04/14 00:19:53 INFO mlflow.tracking.fluent: Experiment with name 'fraud-model-training' does not exist. Creating a new experiment.
2026/04/14 00:20:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/14 00:20:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/14 00:20:09 INFO mlflow.sklearn.utils: Logging the 5 b

🏃 View run bemused-crane-146 at: http://127.0.0.1:8080/#/experiments/1/runs/879a15c0d2b14ccb961d9d4904606128
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1
🏃 View run funny-foal-583 at: http://127.0.0.1:8080/#/experiments/1/runs/cbfa9b7107254507b78324c71871dc83
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1
🏃 View run RandomForestClassifier at: http://127.0.0.1:8080/#/experiments/1/runs/a9525200f55846c780afc54942796fe0
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


In [ ]:
mlflow.sklearn.load_model(best_model_artifact_uri)

In [ ]:
# mlflow.register_model("mlflow-artifacts:/1/models/m-60e99880a7aa43d3becbd945aa6e9038/artifacts", "fraud")

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric